# 2026-11-18-why-fire-stations-have-poles: video-director bundle for Google Colab

Generated by the `google-colab-runner` skill from the manifest `2026-11-18-why-fire-stations-have-poles` (manifest tier **draft**).
This notebook installs ComfyUI and the custom nodes at pinned commits, downloads the model files,
renders every shot's still and video (plus audio jobs), and collects the outputs to Google Drive or a zip.

**Spend.** Running this uses Google Colab compute units for as long as the runtime is connected,
including setup and idle time. Estimates (unverified placeholders; check the Resources panel):

| tier | needs | jobs | session min | compute units |
|---|---|---|---|---|
| draft | 16 GB | 15 | 109.5 | ~3.28 CU on T4 |

A watchdog cell releases the runtime when the session cap is reached, after it has been idle, or
when the director sends a teardown request. You can always stop it yourself: **Runtime > Disconnect and delete runtime**.

**Hugging Face token.** Some model repos are gated. Add your own token as a Google Colab secret named
`HF_TOKEN` (key icon in the left sidebar, enable notebook access). It is read at run time and never written to disk or output.

**Licences.** Model weights are not part of this bundle; each file's licence note is in `models.json`.

In [ ]:
#@title 1. Settings
MODE = "package"  #@param ["package", "orchestrate"]
#@markdown `package`: render the whole bundle here. `orchestrate`: start ComfyUI plus a tunnel and let the director drive it.
TIER = "auto"  #@param ["auto", "draft", "standard", "max"]
OUTPUT = "drive"  #@param ["drive", "zip"]
TUNNEL = "cloudflared"  #@param ["none", "cloudflared"]
MAX_SESSION_MINUTES = 174  #@param {type:"integer"}
IDLE_MINUTES = 15  #@param {type:"integer"}
BUNDLE = "2026-11-18-why-fire-stations-have-poles-google-colab"
ROOT = "/content"
import os, time
SESSION_T0 = time.time()
os.makedirs(f"{ROOT}/{BUNDLE}", exist_ok=True)
print("settings ok:", MODE, TIER, OUTPUT, TUNNEL, MAX_SESSION_MINUTES, "min cap")

In [ ]:
#@title 2. Unpack the bundle (embedded: workflows, jobs, runner; assets are separate)
import base64, io, zipfile, os, glob
PAYLOAD = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(f"{ROOT}/{BUNDLE}")
# Assets (reference images, music) travel in 2026-11-18-why-fire-stations-have-poles-google-colab-assets.zip: upload it to /content or to MyDrive.
for cand in [f"{ROOT}/2026-11-18-why-fire-stations-have-poles-google-colab-assets.zip", "/content/drive/MyDrive/2026-11-18-why-fire-stations-have-poles-google-colab-assets.zip"]:
    if os.path.isfile(cand):
        zipfile.ZipFile(cand).extractall(f"{ROOT}/{BUNDLE}"); print("assets from", cand)
import json
JOBS = json.load(open(f"{ROOT}/{BUNDLE}/jobs.json"))
if JOBS["missing_assets"]: print("WARNING: assets not in the bundle:", JOBS["missing_assets"])
print("bundle ready:", sorted(os.listdir(f"{ROOT}/{BUNDLE}")))

In [ ]:
#@title 3. Detect the GPU and pick the tier
import subprocess
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > pick a GPU (T4, L4, A100 or H100), then re-run.")
GPU_NAME, mib = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
VRAM_GB = round(float(mib) / 1024)
ORDER = ["draft", "standard", "max"]
fits = [t for t in ORDER if t in JOBS["tiers"] and JOBS["tiers"][t]["min_vram_gb"] <= VRAM_GB] or ["draft"]
CHOSEN = fits[-1] if TIER == "auto" else TIER
if ORDER.index(CHOSEN) > ORDER.index(JOBS["manifest_tier"]): CHOSEN = JOBS["manifest_tier"]
print(f"{GPU_NAME} {VRAM_GB} GB -> tier {CHOSEN} (manifest tier {JOBS['manifest_tier']})")
for n in JOBS["tiers"][CHOSEN]["notes"]: print("note:", n)

In [ ]:
#@title 4. Install ComfyUI and custom nodes at pinned commits
COMFY = f"{ROOT}/ComfyUI"
NODES = json.load(open(f"{ROOT}/{BUNDLE}/nodes.lock.json"))
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True)
    if r.returncode: print("  (exit", r.returncode, ")")
    return r.returncode
if not os.path.isdir(COMFY):
    sh(f"git clone -q {NODES['comfyui']['repo']} {COMFY}")
sh(f"cd {COMFY} && git fetch -q origin && git checkout -q {NODES['comfyui']['commit']}")
sh(f"cd {COMFY} && pip -q install -r requirements.txt")
for url, sha in NODES["packs"].items():
    d = f"{COMFY}/custom_nodes/" + url.rstrip("/").split("/")[-1]
    if not os.path.isdir(d): sh(f"git clone -q {url} {d}")
    sh(f"cd {d} && git fetch -q origin && git checkout -q {sha}" if sha else f"echo unpinned {url}")
    if os.path.isfile(f"{d}/requirements.txt"): sh(f"pip -q install -r {d}/requirements.txt")
for d in NODES.get("bundled", []):  # node packs shipped inside the skills collection
    sh(f"rm -rf {COMFY}/custom_nodes/{d} && cp -r {ROOT}/{BUNDLE}/custom_nodes/{d} {COMFY}/custom_nodes/{d}")
    if os.path.isfile(f"{COMFY}/custom_nodes/{d}/requirements.txt"): sh(f"pip -q install -r {COMFY}/custom_nodes/{d}/requirements.txt")
print("installed", len(NODES["packs"]), "pinned packs and", len(NODES.get("bundled", [])), "bundled packs")

In [ ]:
#@title 5. Download model files (your own HF token from Google Colab secrets)
import urllib.request, shutil
def hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")
TOKEN = hf_token()
print("HF token:", "found (not shown)" if TOKEN else "none: gated repos will fail; add the HF_TOKEN secret")
MODELS = json.load(open(f"{ROOT}/{BUNDLE}/models.json"))["files"]
need = JOBS["tiers"][CHOSEN]["models"]; skipped = []
for f in need:
    e = MODELS.get(f) or {}
    dst = f"{COMFY}/models/{e.get('dir', 'diffusion_models').split('/')[0]}/{f}" if not e.get("dir", "").startswith("custom_nodes") else f"{COMFY}/{e['dir']}/{f.split('/')[-1]}"
    if os.path.isfile(dst): print("have", f); continue
    if not e.get("url"): skipped.append(f); continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(f"get  {f} ({(e.get('bytes') or 0)/1e9:.1f} GB) licence: {e.get('license')}")
    req = urllib.request.Request(e["url"], headers={"Authorization": f"Bearer {TOKEN}"} if TOKEN else {})
    try:
        with urllib.request.urlopen(req, timeout=120) as r, open(dst + ".part", "wb") as out: shutil.copyfileobj(r, out, 1 << 24)
        os.replace(dst + ".part", dst)
    except Exception as ex:
        print("  FAILED:", type(ex).__name__, getattr(ex, "code", "")); skipped.append(f)
del TOKEN
if skipped: print("NOT downloaded (no URL or failed; place them by hand or the jobs that need them fail):", skipped)

In [ ]:
#@title 6. Start ComfyUI (127.0.0.1:8188) and wait until it answers
import threading
log = open(f"{ROOT}/comfyui.log", "a")
COMFY_PROC = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd=COMFY, stdout=log, stderr=log)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); print("ComfyUI is up"); break
    except Exception: time.sleep(2)
else: raise SystemExit("ComfyUI did not start; see /content/comfyui.log")

In [ ]:
#@title 7. Watchdog: release the runtime at the cap, when idle, or on a teardown request
def _watchdog():
    idle_since = None
    while True:
        time.sleep(30)
        try:
            from google.colab import runtime
        except Exception:
            runtime = None
        why = None
        if (time.time() - SESSION_T0) / 60 >= MAX_SESSION_MINUTES: why = "session cap"
        if os.path.isfile(f"{COMFY}/input/gcr_teardown_request.txt"): why = "teardown request"
        try:
            q = json.load(urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=5))
            busy = q.get("queue_running") or q.get("queue_pending")
        except Exception:
            busy = False
        idle_since = None if busy else (idle_since or time.time())
        if idle_since and (time.time() - idle_since) / 60 >= IDLE_MINUTES and not RUNNER_ACTIVE: why = "idle"
        if why:
            print(f"watchdog: {why}; collecting outputs and releasing the runtime", flush=True)
            try: collect()
            except Exception as ex: print("collect failed:", ex)
            if runtime: runtime.unassign()
            return
RUNNER_ACTIVE = False
def collect():
    out = f"{ROOT}/{BUNDLE}/outputs"
    if not os.path.isdir(out): return None
    z = shutil.make_archive(f"{ROOT}/{BUNDLE}-outputs", "zip", out)
    if OUTPUT == "drive" and os.path.isdir("/content/drive/MyDrive"):
        shutil.copy2(z, "/content/drive/MyDrive/"); print("copied to Drive:", os.path.basename(z))
    return z
if OUTPUT == "drive":
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as ex: print("Drive not mounted:", ex)
threading.Thread(target=_watchdog, daemon=True).start()
print(f"watchdog on: cap {MAX_SESSION_MINUTES} min, idle {IDLE_MINUTES} min")

In [ ]:
#@title 8. (orchestrate mode) Open a tunnel so the director can reach ComfyUI
#@markdown Anyone with the tunnel URL can submit jobs to this ComfyUI while it is up. Share it only with your agent.
TUNNEL_URL = None
if MODE == "orchestrate" and TUNNEL == "cloudflared":
    if not os.path.isfile("/usr/local/bin/cloudflared"):
        sh("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared")
    import re
    tlog = open(f"{ROOT}/tunnel.log", "w")
    TUN = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"], stdout=tlog, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2); m_ = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{ROOT}/tunnel.log").read())
        if m_: TUNNEL_URL = m_.group(0); break
    print("Give this URL to the director:", TUNNEL_URL)
    print("  python skills/video-director/scripts/direct.py run <manifest> --target google-colab --url", TUNNEL_URL)
else:
    print("no tunnel (MODE is package, or TUNNEL is none)")

In [ ]:
#@title 9. (package mode) Render every job in the bundle (resumable)
if MODE == "package":
    RUNNER_ACTIVE = True
    try:
        rc = subprocess.run(["python", f"{ROOT}/{BUNDLE}/runner.py", "--tier", CHOSEN, "--out", f"{ROOT}/{BUNDLE}/outputs",
                             "--max-minutes", str(max(1, MAX_SESSION_MINUTES - (time.time() - SESSION_T0) / 60 - 3))]).returncode
    finally:
        RUNNER_ACTIVE = False
    print("runner exit", rc, "(0 = all done, 4 = some jobs failed or blocked, 5 = ComfyUI unreachable)")
else:
    print("orchestrate mode: the director submits the jobs; leave this notebook open")

In [ ]:
#@title 10. Collect outputs (Drive or download)
z = collect()
if z and OUTPUT == "zip":
    from google.colab import files; files.download(z)
print("outputs:", z, "- back on your machine: python skills/google-colab-runner/scripts/colab.py ingest <manifest> <outputs.zip>")

In [ ]:
#@title 11. Tear down: release the runtime now (stops compute-unit use)
from google.colab import runtime
runtime.unassign()